In [1]:
import sys
print(sys.executable)

c:\Program Files\Python313\python.exe


In [2]:
!pip install gensim numpy pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from gensim.models import KeyedVectors

print("Загрузка модели...")
model = KeyedVectors.load_word2vec_format(
    r"c:\Users\Анастасия Андреевна\Downloads\диплом_4 курс\models\embeddings\wiki.sr.vec",
    binary=False
)

print("Размерность:", model.vector_size)

print("Вектор biti (первые 10 чисел):")
print(model["biti"][:10])

Загрузка модели...
Размерность: 300
Вектор biti (первые 10 чисел):
[-0.09271   0.26164  -0.61144   0.34113  -0.032337  0.76376   0.16553
 -0.4697    0.054387 -0.14561 ]


In [39]:
import pandas as pd

df = pd.read_csv(
    r"c:\Users\Анастасия Андреевна\Downloads\диплом_4 курс\data\csv\serbian_verb_final_02_02_with_ruscorpora.csv"
)

print(df.head())
print("Всего строк:", len(df))

  base_verb prefixed_verb prefix  freq_base  freq_prefixed
0   gledati     izgledati     iz    1657248           9884
1    činiti       učiniti      u     277966          13331
2    pitati       upitati      u      60569          25040
3  govoriti    odgovoriti     od    1501894           9023
4    laziti      nalaziti     na       1201         120677
Всего строк: 1980


Смотрим, что совпадает в датасете и в эмбеддингах

In [40]:
pairs = []

for _, row in df.iterrows():
    base = row["base_verb"]
    pref = row["prefixed_verb"]
    prefix = row["prefix"]

    if base in model and pref in model:
        pairs.append((base, pref, prefix))

print("Подходящих пар:", len(pairs))

Подходящих пар: 186


In [41]:
from collections import defaultdict

prefix_groups = defaultdict(list)

for base, pref, prefix in pairs:
    prefix_groups[prefix].append((base, pref))

print("Приставки:", list(prefix_groups.keys()))

Приставки: ['iz', 'u', 'od', 'na', 'do', 'za', 'pro', 'sa', 'pre', 'pri', 'uz', 'ob', 'raz', 'pod', 'nad']


In [42]:
print(prefix_groups)

defaultdict(<class 'list'>, {'iz': [('gledati', 'izgledati'), ('gubiti', 'izgubiti'), ('govoriti', 'izgovoriti'), ('vaditi', 'izvaditi'), ('vesti', 'izvesti'), ('držati', 'izdržati'), ('vršiti', 'izvršiti'), ('baciti', 'izbaciti'), ('dati', 'izdati'), ('nositi', 'iznositi'), ('voditi', 'izvoditi'), ('reći', 'izreći'), ('davati', 'izdavati'), ('brisati', 'izbrisati'), ('graditi', 'izgraditi'), ('raditi', 'izraditi'), ('lečiti', 'izlečiti'), ('ostati', 'izostati'), ('računati', 'izračunati'), ('meriti', 'izmeriti'), ('ostaviti', 'izostaviti'), ('glasati', 'izglasati')], 'u': [('činiti', 'učiniti'), ('pitati', 'upitati'), ('gledati', 'ugledati'), ('raditi', 'uraditi'), ('biti', 'ubiti'), ('kazati', 'ukazati'), ('dati', 'udati'), ('tvrditi', 'utvrditi'), ('vesti', 'uvesti'), ('pasti', 'upasti'), ('nositi', 'unositi'), ('baciti', 'ubaciti'), ('pisati', 'upisati'), ('videti', 'uvideti'), ('slediti', 'uslediti'), ('skratiti', 'uskratiti'), ('porediti', 'uporediti'), ('voditi', 'uvoditi'), ('g

Обучение матрицы

In [43]:
import numpy as np
from sklearn.cross_decomposition import PLSRegression

def train_matrix(pairs, n_components=50):
    X = []
    Y = []
    for base, pref in pairs:
        X.append(model[base])
        Y.append(model[pref])

    X = np.array(X)
    Y = np.array(Y)

    n_components = min(n_components, X.shape[0], X.shape[1])
    pls = PLSRegression(n_components=n_components)
    pls.fit(X, Y)
    M = pls.coef_
    return M


In [44]:
def split_pairs(pairs, test_ratio=0.2, seed=42):
    rnd = random.Random(seed)
    pairs = pairs.copy()
    rnd.shuffle(pairs)
    split = int(len(pairs) * (1 - test_ratio))
    return pairs[:split], pairs[split:]

In [45]:
from numpy.linalg import norm

def cosine(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

In [46]:
import random
prefix_results = {}
prefix_matrices = {}

for prefix, group in prefix_groups.items():
    if len(group) >= 10:
        train_pairs, test_pairs = split_pairs(group, test_ratio=0.2)
        M = train_matrix(train_pairs)

        scores = []
        for base, pref in test_pairs:
            predicted = model[base] @ M
            score = cosine(predicted, model[pref])
            scores.append(score)

        mean_score = sum(scores) / len(scores) if scores else 0
        prefix_results[prefix] = mean_score
        prefix_matrices[prefix] = M
        print(f"{prefix}: mean cosine = {mean_score:.3f} (pairs: {len(group)})")


iz: mean cosine = 0.215 (pairs: 22)
u: mean cosine = 0.334 (pairs: 25)
na: mean cosine = 0.271 (pairs: 24)
do: mean cosine = 0.278 (pairs: 16)
za: mean cosine = 0.216 (pairs: 18)
pro: mean cosine = 0.152 (pairs: 17)
pre: mean cosine = 0.137 (pairs: 18)
pri: mean cosine = 0.144 (pairs: 12)


C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 16
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 19
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 18
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 11
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\

In [47]:
prefix_matrices = {}

for prefix, group in prefix_groups.items():
    if len(group) >= 5:  # минимум 5 пар
        M = train_matrix(group)
        prefix_matrices[prefix] = M
        print(f"Обучена матрица для {prefix}, пар: {len(group)}")

Обучена матрица для iz, пар: 22
Обучена матрица для u, пар: 25
Обучена матрица для od, пар: 9
Обучена матрица для na, пар: 24
Обучена матрица для do, пар: 16
Обучена матрица для za, пар: 18
Обучена матрица для pro, пар: 17
Обучена матрица для sa, пар: 9
Обучена матрица для pre, пар: 18
Обучена матрица для pri, пар: 12
Обучена матрица для ob, пар: 6


C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 21
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 24
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 8
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\cross_decomposition\_pls.py:308: UserWarning: y residual is constant at iteration 23
  warnings.warn(f"y residual is constant at iteration {k}")
C:\Users\Анастасия Андреевна\AppData\Roaming\Python\Python313\site-packages\sklearn\c

In [48]:
from numpy.linalg import norm

def cosine(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

base = "biti"
true_pref = "dobiti"

M_do = prefix_matrices["do"]

predicted = model[base] @ M_do

print("Cosine(predicted, true) =",
      cosine(predicted, model[true_pref]))

Cosine(predicted, true) = 0.27747583852733304


In [49]:
import numpy as np
from sklearn.cross_decomposition import PLSRegression

def train_matrix(pairs, n_components=50):
    X = []
    Y = []
    for base, pref in pairs:
        X.append(model[base])
        Y.append(model[pref])

    X = np.array(X)
    Y = np.array(Y)

    n_components = min(n_components, X.shape[0], X.shape[1])
    pls = PLSRegression(n_components=n_components)
    pls.fit(X, Y)
    M = pls.coef_
    return M


In [50]:
import pandas as pd

df = pd.read_csv(r"data\csv\serbian_verb_final_02_02_with_ruscorpora_500.csv")

df = df[["base_verb", "prefixed_verb", "prefix"]]
df = df.dropna()

In [51]:
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format(
    r"models\embeddings\wiki.sr.vec",
    binary=False
)

In [52]:
! pip install https://github.com/inducer/pyfasttext/releases/download/v0.2.6/fasttext-0.9.2-cp313-cp313-win_amd64.whl

Defaulting to user installation because normal site-packages is not writeable


  ERROR: HTTP error 404 while getting https://github.com/inducer/pyfasttext/releases/download/v0.2.6/fasttext-0.9.2-cp313-cp313-win_amd64.whl

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not install requirement fasttext==0.9.2 from https://github.com/inducer/pyfasttext/releases/download/v0.2.6/fasttext-0.9.2-cp313-cp313-win_amd64.whl because of HTTP error 404 Client Error: Not Found for url: https://github.com/inducer/pyfasttext/releases/download/v0.2.6/fasttext-0.9.2-cp313-cp313-win_amd64.whl for URL https://github.com/inducer/pyfasttext/releases/download/v0.2.6/fasttext-0.9.2-cp313-cp313-win_amd64.whl


In [54]:
from gensim.models import KeyedVectors

model = KeyedVectors.load_word2vec_format(
    r"models\embeddings\wiki.sr.vec",
    binary=False,
    limit=200000
)

print("Размер словаря:", len(model))
print("Размер вектора:", model.vector_size)

word = "добар"
vector = model[word]
print(f"Вектор для '{word}':", vector[:10], "...")

similar = model.most_similar("добар", topn=5)
print("Похожие слова:", similar)

Размер словаря: 200000
Размер вектора: 300
Вектор для 'добар': [-0.38532   0.25065  -0.44661  -0.025351 -0.44015  -0.14161   0.053011
 -0.35729  -0.11816   0.12997 ] ...
Похожие слова: [('одличан', 0.7265673875808716), ('добар/сјајан', 0.7129896283149719), ('„добар', 0.6736277341842651), ('сјајан', 0.6712291240692139), ('квалитетан', 0.6491509079933167)]


In [26]:
import pandas as pd

df = pd.read_csv(r"data\csv\serbian_verb_final_02_02_with_ruscorpora_500.csv")

df.head()

,base_verb,prefixed_verb,prefix,freq_base,freq_prefixed,Unnamed: 5,Unnamed: 6
0,biti,dobiti,do,38194575,82730,biti,NaN
1,biti,ubiti,u,38194575,3112,NaN,NaN
2,biti,odbiti,od,38194575,2190,NaN,NaN
3,biti,izbiti,iz,38194575,881,NaN,NaN
4,biti,razbiti,raz,38194575,757,NaN,NaN


In [27]:
missing = []

for _, row in df.iterrows():
    if row["base_verb"] not in model:
        missing.append(row["base_verb"])
    if row["prefixed_verb"] not in model:
        missing.append(row["prefixed_verb"])

print(set(missing))

{'naići', 'ustajati', 'upitati', 'piti', 'sahraniti', 'ulaziti', 'ticati', 'dočekati', 'donositi', 'načiniti', 'mahnuti', 'odložiti', 'izlaziti', 'razbiti', 'zaslužiti', 'uslediti', 'ulagati', 'sušiti', 'hraniti', 'pretrpeti', 'podsticati', 'veriti', 'ložiti', 'beležiti', 'paliti', 'izgovoriti', 'obići', 'moliti', 'slaviti', 'izložiti', 'saslušati', 'sagledati', 'iznositi', 'učiti', 'začuditi', 'uzvratiti', 'graditi', 'paziti', 'buditi', 'izreći', 'udati', 'zabeležiti', 'štititi', 'nadgledati', 'radovati', 'ustati', 'sazvati', 'propustiti', 'izbiti', 'prihvatati', 'začuti', 'prestajati', 'susretati', 'gasiti', 'dužiti', 'sastaviti', 'promrmljati', 'odreći', 'napuniti', 'doznati', 'privući', 'zateći', 'metati', 'dosetiti', 'umiti', 'družiti', 'zagledati', 'najaviti', 'prinositi', 'provoditi', 'podnositi', 'zalagati', 'ličiti', 'ukazivati', 'odlikovati', 'kidati', 'suditi', 'miriti', 'priličiti', 'ljubiti', 'sticati', 'odlaziti', 'ceniti', 'zastupati', 'puštati', 'upasti', 'sećati', 'zap

In [28]:
df["similarity"] = df.apply(
    lambda row: model.similarity(row["base_verb"], row["prefixed_verb"])
    if row["base_verb"] in model and row["prefixed_verb"] in model
    else None,
    axis=1
)

In [29]:
df.groupby("prefix")["similarity"].mean().sort_values(ascending=False)

prefix
raz    0.911533
sa     0.893638
u      0.885761
za     0.874494
od     0.869440
uz     0.864842
na     0.862197
iz     0.858635
pod    0.833341
pro    0.826830
ob     0.817751
do     0.809774
pre    0.797777
pri    0.773807
nad         NaN
su          NaN
Name: similarity, dtype: float64

In [30]:
df[df["prefix"] == "raz"].sort_values("similarity")

,base_verb,prefixed_verb,prefix,freq_base,freq_prefixed,Unnamed: 5,Unnamed: 6,similarity
40,misliti,razmisliti,raz,594430,1233,NaN,NaN,0.911533
4,biti,razbiti,raz,38194575,757,NaN,NaN,NaN
148,viti,razviti,raz,26814,3112,viti,NaN,NaN
181,vijati,razvijati,raz,8657,4633,NaN,NaN,NaN
225,likovati,razlikovati,raz,3376,2626,likovati,NaN,NaN
274,motriti,razmotriti,raz,709,3328,motriti,NaN,NaN


In [31]:
# Список глаголов
verbs = ["ходати", "радити", "спавати", "јести", "пити", "трчати", "писати", "читати", "говорити", "слушати"]

for verb in verbs:
    print(f"\nГлагол: {verb}")
    vector = model[verb]
    print("Вектор (первые 10 значений):", vector[:10])
    
    # 5 наиболее похожих слов
    similar = model.most_similar(verb, topn=5)
    print("Похожие слова:", similar)


Глагол: ходати
Вектор (первые 10 значений): [-0.14547   0.46615   0.26156  -0.25031  -0.39951  -0.18033   0.18389
 -0.59898  -0.037136  0.51431 ]
Похожие слова: [('одати', 0.6964132785797119), ('ходали', 0.6543284058570862), ('трчати', 0.6337757110595703), ('ходао', 0.6324000358581543), ('спавати', 0.6238958835601807)]

Глагол: радити
Вектор (первые 10 значений): [-0.12427   0.14142  -0.31509   0.036592 -0.5587   -0.36224  -0.14518
 -0.033341 -0.19702   0.32965 ]
Похожие слова: [('порадити', 0.7760341763496399), ('урадити', 0.7692096829414368), ('одрадити', 0.7626826763153076), ('дорадити', 0.7256090641021729), ('зарадити', 0.6932438611984253)]

Глагол: спавати
Вектор (первые 10 значений): [ 0.012006  0.3656   -0.041185 -0.1708   -0.6845   -0.20264   0.11738
 -0.49001  -0.12699   0.4845  ]
Похожие слова: [('спава', 0.7244544625282288), ('свати', 0.697878360748291), ('заспати', 0.6949657797813416), ('спавају', 0.6919047236442566), ('дешавати', 0.6900447607040405)]

Глагол: јести
Вектор

In [33]:
import pickle
from pprint import pprint

with open(r"models\artifacts\prefix_matrices_cc_partial.pkl", "rb") as f:
    art = pickle.load(f)

print(art.keys())
pprint(art["config"])
print("prefixes:", list(art["W_prefix"].keys()))
print("W_global shape:", art["W_global"].shape)


dict_keys(['config', 'W_global', 'W_prefix', 'prefix_stats', 'holdout_examples', 'metrics'])
{'alpha': 10.0,
 'dataset': 'serbian_verb_final_02_02_with_ruscorpora_500.csv',
 'dim': 300,
 'dropped_rows': 62,
 'embedding_mode': 'selected_vec',
 'embeddings': 'cc.sr.300.vec',
 'loaded_word_vectors': 373,
 'min_pairs': 8,
 'seed': 42,
 'test_ratio': 0.2,
 'test_rows': 43,
 'train_rows': 175,
 'trained_prefixes': 10}
prefixes: ['do', 'iz', 'na', 'od', 'pre', 'pri', 'pro', 'sa', 'u', 'za']
W_global shape: (300, 300)
